In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import os
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')

# Plot configuration
plt.style.use('default')
sns.set_palette("Set2")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# Display configuration
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.notebook_repr_html', True)

In [ ]:
base_path = Path("../60_analyses/csv/qualitative/exp2/students")

# Path configuration like evaluation1
BASE_PROJECT_PATH = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
hint_base_path = os.path.join(BASE_PROJECT_PATH, "20_experiments/60_analyses/csv/qualitative/exp2/hints")
output_base_path = os.path.join(BASE_PROJECT_PATH, "40_evaluation/exp2/qualitative")
output_tables_path = os.path.join(output_base_path, "tables")
output_plots_path = os.path.join(output_base_path, "plots")

for path in [output_tables_path, output_plots_path]:
    os.makedirs(path, exist_ok=True)

# Storage for results
tables = {}
plots = {}

# Define overlap configuration per student
overlap_config = {
    "student_1": {
        "exp2a": range(1, 17),   # Questions 1-16
        "exp2b": range(1, 5),    # Questions 1-4
        "exp2c": []
    },
    "student_2": {
        "exp2a": range(11, 17),  # Questions 11-16  
        "exp2b": range(1, 9),    # Questions 1-8
        "exp2c": range(1, 7)     # Questions 1-6
    },
    "student_3": {
        "exp2a": [],
        "exp2b": range(5, 9),    # Questions 5-8
        "exp2c": range(1, 17)    # Questions 1-16
    }
}

print("Setup completed successfully")
print(f"Output tables: {output_tables_path}")
print(f"Output plots: {output_plots_path}")

In [ ]:
def load_student_data(student_id):
    student_path = base_path / f"student_{student_id}"
    data = {}
    
    for exp in ["exp2a", "exp2b", "exp2c"]:
        csv_path = student_path / f"{exp}.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            
            # Filter by overlap configuration
            questions = overlap_config[f"student_{student_id}"][exp]
            if questions:
                df['sample_id'] = df['sample_id'].astype(str)
                df['question_num'] = df['sample_id'].str.extract('(\d+)').astype(int).iloc[:, 0]
                df = df[df['question_num'].isin(questions)]
                df['student'] = student_id
                df['experiment'] = exp
                data[exp] = df
    
    return data

def load_hint_data():
    hint_data = {}
    for exp in ["exp2a", "exp2b", "exp2c"]:
        hint_path = os.path.join(hint_base_path, f"{exp}_hints.csv")
        if os.path.exists(hint_path):
            hint_df = pd.read_csv(hint_path)
            # Create sample_id from question_id for matching
            hint_df['sample_id'] = hint_df['question_id'].astype(str).str.zfill(3)
            hint_data[exp] = hint_df
    return hint_data

# Load all student data
all_data = {}
for i in range(1, 4):
    all_data[f"student_{i}"] = load_student_data(i)

# Load hint data
hint_data = load_hint_data()
print(f"Loaded hint data for experiments: {list(hint_data.keys())}")

In [ ]:
def combine_data():
    combined = []
    
    for student, experiments in all_data.items():
        for exp, df in experiments.items():
            if not df.empty:
                # Add hint data (llm, question_type) 
                if exp in hint_data:
                    hint_df = hint_data[exp]
                    # Merge on sample_id
                    df = df.merge(hint_df[['sample_id', 'llm', 'question_type']], 
                                 on='sample_id', how='left')
                combined.append(df)
    
    if combined:
        return pd.concat(combined, ignore_index=True)
    return pd.DataFrame()

df_combined = combine_data()
print(f"Total evaluations: {len(df_combined)}")
print(f"Questions per student: {df_combined.groupby('student').size()}")
print(f"Questions per experiment: {df_combined.groupby('experiment').size()}")

# Check LLM distribution
if 'llm' in df_combined.columns:
    print(f"Questions per LLM: {df_combined.groupby('llm').size()}")
    print(f"Available LLMs: {list(df_combined['llm'].unique())}")
    
if 'question_type' in df_combined.columns:
    print(f"Questions per type: {df_combined.groupby('question_type').size()}")

In [ ]:
def calc_stats(df, metric):
    numeric_data = pd.to_numeric(df[metric], errors='coerce')
    return {
        'mean': numeric_data.mean(),
        'std': numeric_data.std(),
        'count': numeric_data.count()
    }

def overlap_analysis():
    results = {}
    
    # Questions with multiple evaluations (overlaps)
    overlap_questions = []
    
    # Find overlaps between students
    for exp in ["exp2a", "exp2b", "exp2c"]:
        exp_data = df_combined[df_combined['experiment'] == exp]
        question_counts = exp_data['sample_id'].value_counts()
        overlap_questions.extend(question_counts[question_counts > 1].index.tolist())
    
    results['overlap_questions'] = overlap_questions
    results['total_overlaps'] = len(overlap_questions)
    
    return results

overlap_info = overlap_analysis()
print(f"Questions with overlapping evaluations: {overlap_info['total_overlaps']}")

In [ ]:
def inter_rater_reliability():
    metrics = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'bloom_rating']
    reliability_results = {}
    
    for metric in metrics:
        agreements = []
        
        for q_id in overlap_info['overlap_questions']:
            q_data = df_combined[df_combined['sample_id'] == q_id]
            scores = pd.to_numeric(q_data[metric], errors='coerce').dropna()
            
            if len(scores) >= 2:
                # Calculate agreement (simple correlation for now)
                if len(scores.unique()) > 1:
                    agreements.append(scores.std())
                else:
                    agreements.append(0)  # Perfect agreement
        
        reliability_results[metric] = {
            'avg_std': np.mean(agreements) if agreements else 0,
            'num_comparisons': len(agreements)
        }
    
    return reliability_results

reliability = inter_rater_reliability()

for metric, stats in reliability.items():
    print(f"{metric}: avg std = {stats['avg_std']:.3f}, comparisons = {stats['num_comparisons']}")

In [ ]:
def plot_metrics():
    metrics = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'bloom_rating']
    
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    axes = axes.flatten()
    
    for i, metric in enumerate(metrics):
        numeric_data = pd.to_numeric(df_combined[metric], errors='coerce')
        df_plot = df_combined.copy()
        df_plot[metric] = numeric_data
        
        sns.boxplot(data=df_plot, x='experiment', y=metric, ax=axes[i])
        axes[i].set_title(f'{metric.title()}')
        axes[i].tick_params(axis='x', rotation=45)
    
    # Hide the last two empty subplots
    axes[7].set_visible(False)
    axes[8].set_visible(False)
    
    plt.tight_layout()
    plt.show()

def summary_stats():
    metrics = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'bloom_rating']
    
    summary = {}
    for exp in ["exp2a", "exp2b", "exp2c"]:
        exp_data = df_combined[df_combined['experiment'] == exp]
        summary[exp] = {}
        
        for metric in metrics:
            summary[exp][metric] = calc_stats(exp_data, metric)
    
    return summary

stats = summary_stats()
plot_metrics()

In [ ]:
def create_seaborn_boxplot(data, x, y, ax, title, ylabel, xlabel, scale_range=None):
    colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3', '#a6d854', '#ffd92f']
    unique_vals = sorted(data[x].unique())
    palette = colors[:len(unique_vals)]
    
    sns.boxplot(data=data, x=x, y=y, ax=ax, palette=palette,
                medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
    
    for i, val in enumerate(unique_vals):
        mean_val = data[data[x] == val][y].mean()
        ax.scatter(i, mean_val, color='red', marker='D', s=50, zorder=3, 
                  edgecolor='darkred', linewidth=1)
    
    label_mapping = {
        'exp2a': 'Type Only', 'exp2b': 'Bloom Only', 'exp2c': 'Type + Bloom'
    }
    
    current_labels = [tick.get_text() for tick in ax.get_xticklabels()]
    new_labels = [label_mapping.get(label, label) for label in current_labels]
    ax.set_xticklabels(new_labels, rotation=0)
    
    if scale_range:
        ax.set_ylim(scale_range)
        if scale_range == (0, 10):
            title += " (0-10 scale)"
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11, labelpad=15)
    ax.set_xlabel(xlabel, fontsize=11, labelpad=10)
    ax.grid(True, alpha=0.3)

# Experiment 2: Descriptive Statistics

Analysis of prompt engineering approaches for question generation.

In [ ]:
print("EXPERIMENT 2 - DESCRIPTIVE STATISTICS")
print("="*60)

metrics = ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'bloom_rating']

# Overall statistics
exp2_stats = df_combined[metrics].describe().round(2)
tables['exp2_overall_stats'] = exp2_stats
print("\nOverall Statistics:")
display(exp2_stats)

# Statistics by Experiment 
exp2_exp_stats = df_combined.groupby('experiment')[metrics].agg(['mean', 'std', 'median', 'count']).round(2)
tables['exp2_exp_stats'] = exp2_exp_stats
print("\nStatistics by Experiment:")
display(exp2_exp_stats)

# Experiment ranking
exp2_exp_means = df_combined.groupby('experiment')[metrics].mean().round(2)
exp2_exp_overall = exp2_exp_means.mean(axis=1).sort_values(ascending=False)
tables['exp2_exp_ranking'] = exp2_exp_overall

print("\nOverall Experiment Ranking:")
exp_names = {'exp2a': 'Type Only', 'exp2b': 'Bloom Only', 'exp2c': 'Type + Bloom'}
for i, (exp, score) in enumerate(exp2_exp_overall.items(), 1):
    print(f"{i}. {exp_names[exp]}: {score:.2f}")

# Statistics by Student
exp2_student_stats = df_combined.groupby('student')[metrics].agg(['mean', 'std', 'median', 'count']).round(2)
tables['exp2_student_stats'] = exp2_student_stats
print("\nStatistics by Student:")
display(exp2_student_stats)

# Statistics by LLM (if available)
if 'llm' in df_combined.columns:
    exp2_llm_stats = df_combined.groupby('llm')[metrics].agg(['mean', 'std', 'median', 'count']).round(2)
    tables['exp2_llm_stats'] = exp2_llm_stats
    print("\nStatistics by LLM:")
    display(exp2_llm_stats)
    
    # LLM ranking
    exp2_llm_means = df_combined.groupby('llm')[metrics].mean().round(2)
    exp2_llm_overall = exp2_llm_means.mean(axis=1).sort_values(ascending=False)
    tables['exp2_llm_ranking'] = exp2_llm_overall
    
    print("\nOverall LLM Ranking:")
    for i, (llm, score) in enumerate(exp2_llm_overall.items(), 1):
        print(f"{i}. {llm.title()}: {score:.2f}")

# Statistics by Question Type (if available)
if 'question_type' in df_combined.columns:
    exp2_qtype_stats = df_combined.groupby('question_type')[metrics].agg(['mean', 'std', 'median', 'count']).round(2)
    tables['exp2_qtype_stats'] = exp2_qtype_stats
    print("\nStatistics by Question Type:")
    display(exp2_qtype_stats)

In [ ]:
# Experiment Performance Visualization
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
axes = axes.flatten()
plots['exp2_experiment_analysis'] = fig

for i, metric in enumerate(metrics):
    df_plot = df_combined.copy()
    df_plot[metric] = pd.to_numeric(df_plot[metric], errors='coerce')
    df_plot = df_plot.dropna(subset=[metric])
    
    create_seaborn_boxplot(df_plot, 'experiment', metric, axes[i], 
                          f'{metric.title()}', metric.title(), 'Experiment', scale_range=(0, 10))

# Hide the last two empty subplots
axes[7].set_visible(False)
axes[8].set_visible(False)

plt.suptitle('Experiment 2: Performance across Prompt Engineering Approaches', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

In [ ]:
# Student Performance Analysis
fig, axes = plt.subplots(3, 3, figsize=(18, 16))
axes = axes.flatten()
plots['exp2_student_analysis'] = fig

for i, metric in enumerate(metrics):
    df_plot = df_combined.copy()
    df_plot[metric] = pd.to_numeric(df_plot[metric], errors='coerce')
    df_plot = df_plot.dropna(subset=[metric])
    
    colors = ['#66c2a5', '#fc8d62', '#8da0cb']
    
    sns.boxplot(data=df_plot, x='student', y=metric, ax=axes[i], palette=colors,
                medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
    
    for j, student in enumerate(sorted(df_plot['student'].unique())):
        mean_val = df_plot[df_plot['student'] == student][metric].mean()
        axes[i].scatter(j, mean_val, color='red', marker='D', s=50, zorder=3, 
                       edgecolor='darkred', linewidth=1)
    
    axes[i].set_ylim(0, 10)
    axes[i].set_title(f'{metric.title()} (0-10 scale)', fontsize=12, fontweight='bold')
    axes[i].set_ylabel(metric.title(), fontsize=11, labelpad=15)
    axes[i].set_xlabel('Student', fontsize=11, labelpad=10)
    axes[i].grid(True, alpha=0.3)

# Hide the last two empty subplots
axes[7].set_visible(False)
axes[8].set_visible(False)

plt.suptitle('Experiment 2: Performance by Student Evaluator', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.93)
plt.show()

In [ ]:
# LLM Performance Analysis
if 'llm' in df_combined.columns:
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    axes = axes.flatten()
    plots['exp2_llm_analysis'] = fig

    for i, metric in enumerate(metrics):
        df_plot = df_combined.copy()
        df_plot[metric] = pd.to_numeric(df_plot[metric], errors='coerce')
        df_plot = df_plot.dropna(subset=[metric])
        
        colors = ['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3']
        
        sns.boxplot(data=df_plot, x='llm', y=metric, ax=axes[i], palette=colors,
                    medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
        
        for j, llm in enumerate(sorted(df_plot['llm'].unique())):
            mean_val = df_plot[df_plot['llm'] == llm][metric].mean()
            axes[i].scatter(j, mean_val, color='red', marker='D', s=50, zorder=3, 
                           edgecolor='darkred', linewidth=1)
        
        axes[i].set_ylim(0, 10)
        axes[i].set_title(f'{metric.title()} (0-10 scale)', fontsize=12, fontweight='bold')
        axes[i].set_ylabel(metric.title(), fontsize=11, labelpad=15)
        axes[i].set_xlabel('LLM', fontsize=11, labelpad=10)
        axes[i].grid(True, alpha=0.3)
        axes[i].tick_params(axis='x', rotation=45)

    # Hide the last two empty subplots
    axes[7].set_visible(False)
    axes[8].set_visible(False)

    plt.suptitle('Experiment 2: Performance by LLM', 
                 fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()
else:
    print("LLM data not available for visualization")

In [ ]:
# Question Type Performance Analysis
if 'question_type' in df_combined.columns:
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    axes = axes.flatten()
    plots['exp2_qtype_analysis'] = fig

    for i, metric in enumerate(metrics):
        df_plot = df_combined.copy()
        df_plot[metric] = pd.to_numeric(df_plot[metric], errors='coerce')
        df_plot = df_plot.dropna(subset=[metric])
        
        colors = ['#a6d854', '#ffd92f']
        
        sns.boxplot(data=df_plot, x='question_type', y=metric, ax=axes[i], palette=colors,
                    medianprops={'color': 'black', 'linewidth': 2.5, 'linestyle': ':'})
        
        for j, qtype in enumerate(sorted(df_plot['question_type'].unique())):
            mean_val = df_plot[df_plot['question_type'] == qtype][metric].mean()
            axes[i].scatter(j, mean_val, color='red', marker='D', s=50, zorder=3, 
                           edgecolor='darkred', linewidth=1)
        
        axes[i].set_ylim(0, 10)
        axes[i].set_title(f'{metric.title()} (0-10 scale)', fontsize=12, fontweight='bold')
        axes[i].set_ylabel(metric.title(), fontsize=11, labelpad=15)
        axes[i].set_xlabel('Question Type', fontsize=11, labelpad=10)
        axes[i].grid(True, alpha=0.3)

    # Hide the last two empty subplots
    axes[7].set_visible(False)
    axes[8].set_visible(False)

    plt.suptitle('Experiment 2: Performance by Question Type', 
                 fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()
else:
    print("Question type data not available for visualization")

In [ ]:
# Heatmap Analysis
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
plots['exp2_heatmaps'] = fig

# Heatmap 1: Mean scores by Experiment vs Metric
heatmap_mean = df_combined.groupby('experiment')[metrics].mean()
heatmap_std = df_combined.groupby('experiment')[metrics].std()

annot_matrix = heatmap_mean.copy()
for i in range(len(heatmap_mean.index)):
    for j in range(len(heatmap_mean.columns)):
        mean_val = heatmap_mean.iloc[i, j]
        std_val = heatmap_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
        elif pd.notna(mean_val):
            annot_matrix.iloc[i, j] = f"{mean_val:.1f}"

exp_labels = ['Type Only', 'Bloom Only', 'Type + Bloom']
metric_labels = [m.title() for m in metrics]

sns.heatmap(heatmap_mean, annot=annot_matrix, fmt='', cmap='RdYlBu_r',
            center=heatmap_mean.values.mean(), square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Score'}, annot_kws={'size': 10, 'weight': 'bold'},
            xticklabels=metric_labels, yticklabels=exp_labels, ax=ax1)

ax1.set_title('Mean Scores by Experiment\nValues: Mean (Std) | Scale: 0-10 points', 
             fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel('Evaluation Criteria', fontsize=12, fontweight='bold')
ax1.set_ylabel('Prompt Approach', fontsize=12, fontweight='bold')

# Heatmap 2: Mean scores by Student vs Experiment
heatmap_student_exp = df_combined.groupby(['student', 'experiment'])[metrics].mean().mean(axis=1).unstack()
heatmap_student_exp_std = df_combined.groupby(['student', 'experiment'])[metrics].std().mean(axis=1).unstack()

annot_matrix_student = heatmap_student_exp.copy()
for i in range(len(heatmap_student_exp.index)):
    for j in range(len(heatmap_student_exp.columns)):
        mean_val = heatmap_student_exp.iloc[i, j]
        std_val = heatmap_student_exp_std.iloc[i, j]
        if pd.notna(mean_val) and pd.notna(std_val):
            annot_matrix_student.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
        elif pd.notna(mean_val):
            annot_matrix_student.iloc[i, j] = f"{mean_val:.1f}"

student_labels = [f'Student {s}' for s in sorted(df_combined['student'].unique())]

sns.heatmap(heatmap_student_exp, annot=annot_matrix_student, fmt='', cmap='RdYlBu_r',
            center=heatmap_student_exp.values.mean(), square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Average Score'}, annot_kws={'size': 10, 'weight': 'bold'},
            xticklabels=exp_labels, yticklabels=student_labels, ax=ax2)

ax2.set_title('Average Scores by Student vs Experiment\nValues: Mean (Std) | Scale: 0-10 points', 
             fontsize=14, fontweight='bold', pad=20)
ax2.set_xlabel('Prompt Approach', fontsize=12, fontweight='bold')
ax2.set_ylabel('Evaluator', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Store detailed statistics tables
exp2_detailed_stats = df_combined.groupby(['experiment', 'student'])[metrics].agg(['mean', 'std', 'median', 'min', 'max', 'count']).round(2)
tables['exp2_detailed_stats'] = exp2_detailed_stats

# LLM-Experiment combinations (if available)
if 'llm' in df_combined.columns:
    exp2_llm_exp_stats = df_combined.groupby(['llm', 'experiment'])[metrics].agg(['mean', 'std', 'median', 'count']).round(2)
    tables['exp2_llm_exp_stats'] = exp2_llm_exp_stats
    
    # Create LLM vs Experiment heatmap
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    plots['exp2_llm_heatmaps'] = fig
    
    # Heatmap 1: LLM vs Experiment (average across all metrics)
    llm_exp_heatmap = df_combined.groupby(['llm', 'experiment'])[metrics].mean().mean(axis=1).unstack()
    llm_exp_std = df_combined.groupby(['llm', 'experiment'])[metrics].std().mean(axis=1).unstack()
    
    annot_matrix_llm = llm_exp_heatmap.copy()
    for i in range(len(llm_exp_heatmap.index)):
        for j in range(len(llm_exp_heatmap.columns)):
            mean_val = llm_exp_heatmap.iloc[i, j]
            std_val = llm_exp_std.iloc[i, j]
            if pd.notna(mean_val) and pd.notna(std_val):
                annot_matrix_llm.iloc[i, j] = f"{mean_val:.1f}\n({std_val:.1f})"
            elif pd.notna(mean_val):
                annot_matrix_llm.iloc[i, j] = f"{mean_val:.1f}"
    
    exp_labels = ['Type Only', 'Bloom Only', 'Type + Bloom']
    llm_labels = [llm.title() for llm in sorted(df_combined['llm'].unique())]
    
    sns.heatmap(llm_exp_heatmap, annot=annot_matrix_llm, fmt='', cmap='RdYlBu_r',
                center=llm_exp_heatmap.values.mean(), square=True, linewidths=0.5,
                cbar_kws={'shrink': 0.8, 'label': 'Average Score'}, annot_kws={'size': 10, 'weight': 'bold'},
                xticklabels=exp_labels, yticklabels=llm_labels, ax=ax1)
    
    ax1.set_title('Average Scores by LLM vs Experiment\nValues: Mean (Std) | Scale: 0-10 points', 
                 fontsize=14, fontweight='bold', pad=20)
    ax1.set_xlabel('Prompt Approach', fontsize=12, fontweight='bold')
    ax1.set_ylabel('LLM', fontsize=12, fontweight='bold')
    
    # Heatmap 2: LLM vs Question Type (if available)
    if 'question_type' in df_combined.columns:
        llm_qtype_heatmap = df_combined.groupby(['llm', 'question_type'])[metrics].mean().mean(axis=1).unstack()
        
        sns.heatmap(llm_qtype_heatmap, annot=True, fmt='.1f', cmap='RdYlBu_r',
                    center=llm_qtype_heatmap.values.mean(), square=True, linewidths=0.5,
                    cbar_kws={'shrink': 0.8, 'label': 'Average Score'}, annot_kws={'size': 10, 'weight': 'bold'},
                    xticklabels=[t.replace('_', ' ').title() for t in llm_qtype_heatmap.columns], 
                    yticklabels=llm_labels, ax=ax2)
        
        ax2.set_title('Average Scores by LLM vs Question Type\nScale: 0-10 points', 
                     fontsize=14, fontweight='bold', pad=20)
        ax2.set_xlabel('Question Type', fontsize=12, fontweight='bold')
        ax2.set_ylabel('LLM', fontsize=12, fontweight='bold')
    else:
        ax2.text(0.5, 0.5, 'Question Type\nData Not Available', 
                ha='center', va='center', transform=ax2.transAxes, fontsize=14)
        ax2.set_xticks([])
        ax2.set_yticks([])
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Comparative Analysis
print("\nCOMPARATIVE ANALYSIS")
print("="*50)

# Experiment comparisons
exp_means = df_combined.groupby('experiment')[metrics].mean().round(2)
tables['exp2_experiment_means'] = exp_means

print("\nExperiment Comparisons:")
print("Type Only vs Bloom Only:")
if 'exp2a' in exp_means.index and 'exp2b' in exp_means.index:
    diff_a_b = exp_means.loc['exp2a'] - exp_means.loc['exp2b']
    for metric, diff in diff_a_b.items():
        direction = "higher" if diff > 0 else "lower"
        print(f"  {metric.title()}: {diff:+.2f} ({direction} for Type Only)")

print("\nType Only vs Type+Bloom:")
if 'exp2a' in exp_means.index and 'exp2c' in exp_means.index:
    diff_a_c = exp_means.loc['exp2a'] - exp_means.loc['exp2c']
    for metric, diff in diff_a_c.items():
        direction = "higher" if diff > 0 else "lower"
        print(f"  {metric.title()}: {diff:+.2f} ({direction} for Type Only)")

print("\nBloom Only vs Type+Bloom:")
if 'exp2b' in exp_means.index and 'exp2c' in exp_means.index:
    diff_b_c = exp_means.loc['exp2b'] - exp_means.loc['exp2c']
    for metric, diff in diff_b_c.items():
        direction = "higher" if diff > 0 else "lower"
        print(f"  {metric.title()}: {diff:+.2f} ({direction} for Bloom Only)")

# Best performing approaches per metric
print("\nBest Performing Approach per Metric:")
for metric in metrics:
    best_exp = exp_means[metric].idxmax()
    best_score = exp_means[metric].max()
    exp_name = {'exp2a': 'Type Only', 'exp2b': 'Bloom Only', 'exp2c': 'Type + Bloom'}[best_exp]
    print(f"  {metric.title()}: {exp_name} ({best_score:.2f})")

# LLM comparisons (if available)
if 'llm' in df_combined.columns:
    print("\n" + "="*50)
    print("LLM COMPARATIVE ANALYSIS")
    print("="*50)
    
    llm_means = df_combined.groupby('llm')[metrics].mean().round(2)
    tables['exp2_llm_means'] = llm_means
    
    print("\nBest Performing LLM per Metric:")
    for metric in metrics:
        best_llm = llm_means[metric].idxmax()
        best_score = llm_means[metric].max()
        print(f"  {metric.title()}: {best_llm.title()} ({best_score:.2f})")
    
    # LLM pairwise comparisons
    llms = list(llm_means.index)
    if len(llms) >= 2:
        print("\nPairwise LLM Comparisons (first vs others):")
        base_llm = llms[0]
        for other_llm in llms[1:]:
            print(f"\n{base_llm.title()} vs {other_llm.title()}:")
            diff = llm_means.loc[base_llm] - llm_means.loc[other_llm]
            for metric, diff_val in diff.items():
                direction = "higher" if diff_val > 0 else "lower"
                print(f"  {metric.title()}: {diff_val:+.2f} ({direction} for {base_llm.title()})")

# Question Type comparisons (if available)
if 'question_type' in df_combined.columns:
    print("\n" + "="*50)
    print("QUESTION TYPE COMPARATIVE ANALYSIS")
    print("="*50)
    
    qtype_means = df_combined.groupby('question_type')[metrics].mean().round(2)
    tables['exp2_qtype_means'] = qtype_means
    
    print("\nMean Scores by Question Type:")
    for qtype, scores in qtype_means.iterrows():
        avg_score = scores.mean()
        print(f"  {qtype.replace('_', ' ').title()}: {avg_score:.2f}")
    
    if len(qtype_means) == 2:
        qtypes = list(qtype_means.index)
        print(f"\n{qtypes[0].title()} vs {qtypes[1].title()}:")
        diff = qtype_means.loc[qtypes[0]] - qtype_means.loc[qtypes[1]]
        for metric, diff_val in diff.items():
            direction = "higher" if diff_val > 0 else "lower"
            print(f"  {metric.title()}: {diff_val:+.2f} ({direction} for {qtypes[0].title()})")

# Inter-Student Reliability Analysis

Analysis of agreement between student evaluators on overlapping questions.

In [ ]:
print("INTER-STUDENT RELIABILITY ANALYSIS")
print("="*50)

# Enhanced reliability analysis
reliability_detailed = {}
correlation_data = []

for metric in metrics:
    agreements = []
    correlations = []
    
    for q_id in overlap_info['overlap_questions']:
        q_data = df_combined[df_combined['sample_id'] == q_id]
        scores = pd.to_numeric(q_data[metric], errors='coerce').dropna()
        
        if len(scores) >= 2:
            if len(scores.unique()) > 1:
                agreements.append(scores.std())
            else:
                agreements.append(0)
            
            # Store for correlation analysis
            for i, score in enumerate(scores):
                correlation_data.append({
                    'metric': metric,
                    'question': q_id,
                    'score': score,
                    'student': q_data.iloc[i]['student'],
                    'experiment': q_data.iloc[i]['experiment']
                })
    
    reliability_detailed[metric] = {
        'avg_std': np.mean(agreements) if agreements else 0,
        'num_comparisons': len(agreements),
        'min_std': np.min(agreements) if agreements else 0,
        'max_std': np.max(agreements) if agreements else 0
    }

reliability_df = pd.DataFrame(reliability_detailed).T.round(3)
tables['exp2_reliability'] = reliability_df

print("\nDetailed Reliability Analysis:")
display(reliability_df)

# Reliability interpretation
print("\nReliability Interpretation (Lower std = Better agreement):")
for metric, stats in reliability_detailed.items():
    level = ("Excellent" if stats['avg_std'] < 0.5 else 
            "Good" if stats['avg_std'] < 1.0 else 
            "Moderate" if stats['avg_std'] < 1.5 else 
            "Poor")
    print(f"  {metric.title()}: {stats['avg_std']:.3f} std ({level})")

# Overlap details by experiment
print("\nOverlap Analysis by Experiment:")
for exp in ["exp2a", "exp2b", "exp2c"]:
    exp_data = df_combined[df_combined['experiment'] == exp]
    overlaps = exp_data['sample_id'].value_counts()
    overlap_count = (overlaps > 1).sum()
    total_questions = len(overlaps)
    print(f"  {exp}: {overlap_count}/{total_questions} questions have overlaps ({100*overlap_count/total_questions:.1f}%)")

In [ ]:
def export_results():
    # Create summary table
    summary_df = pd.DataFrame()
    
    for exp in ["exp2a", "exp2b", "exp2c"]:
        for metric in ['relevance', 'clarity', 'answerability', 'challenging', 'value', 'language', 'bloom_rating']:
            row = {
                'experiment': exp,
                'metric': metric,
                'mean': stats[exp][metric]['mean'],
                'std': stats[exp][metric]['std'],
                'count': stats[exp][metric]['count']
            }
            summary_df = pd.concat([summary_df, pd.DataFrame([row])], ignore_index=True)
    
    # Export files (like evaluation1)
    output_path = Path("../60_analyses/csv/qualitative/exp2/")
    output_path.mkdir(exist_ok=True)
    
    # Export main results
    summary_df.to_csv(output_path / "exp2_summary.csv", index=False)
    df_combined.to_csv(output_path / "exp2_combined.csv", index=False)
    
    return summary_df

def save_all_results():
    print("Saving results...")
    
    # Use student prefix for exp2 (like supervisor/staff prefix in exp1)
    prefix = "student_"
    
    tables_saved = 0
    for table_name, table_data in tables.items():
        csv_path = os.path.join(output_tables_path, f"{prefix}{table_name}.csv")
        table_data.to_csv(csv_path)
        tables_saved += 1
        print(f"Saved table: {prefix}{table_name}.csv")
    
    plots_saved = 0
    for plot_name, plot_fig in plots.items():
        png_path = os.path.join(output_plots_path, f"{prefix}{plot_name}.png")
        plot_fig.savefig(png_path, dpi=300, bbox_inches='tight')
        plots_saved += 1
        print(f"Saved plot: {prefix}{plot_name}.png")
    
    print(f"\nExport Summary:")
    print(f"  Tables saved: {tables_saved}")
    print(f"  Plots saved: {plots_saved}")
    print(f"  Output location: {output_base_path}")
    print(f"  File prefix: {prefix}")

summary_table = export_results()
save_all_results()

print("\nSummary by experiment:")
print(summary_table.pivot_table(values='mean', index='metric', columns='experiment', aggfunc='first').round(3))

print(f"\nCombined evaluations: {len(df_combined)}")
print(f"Overlap questions: {overlap_info['total_overlaps']}")